# End-to-End Pipeline Runner

This notebook is the operational runner for the ChurnGuard pipeline. It drives the library
in `src/` end-to-end: load raw CSVs → preprocess → fold in realtime events → re-derive feature
list → train → tune threshold → save artifacts → sanity-check the reload.

**Typical flow:**

1. Edit a constant in `src/config.py` (or `models/feature_columns.json`)
2. Re-run cell 1 (module reload)
3. Re-run cells 2 → 3 (realtime) → 4 (selection) → 5 → 6 on every pass — realtime events
   grow the `rt_*` candidate pool every run, so feature selection is no longer skippable
4. Cell 7 confirms the saved artifacts round-trip

## Module reload

Run this after any edit to src/config.py or src/*.py. Notebooks cache imports.

In [59]:
# Resolve project root and make src/ importable + relative paths work.
# Jupyter's CWD defaults to the notebook's directory; we want the project root
# so `import src.*` resolves and `data/raw/...` / `models/...` resolve unchanged.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("cwd:", Path.cwd())

import importlib
import src.config, src.preprocessing, src.features, src.predict
for m in (src.config, src.preprocessing, src.features, src.predict):
    importlib.reload(m)
print("reloaded:", [m.__name__ for m in (src.config, src.preprocessing, src.features, src.predict)])


cwd: /Users/vidyutveedgav/churnguard-analytics
reloaded: ['src.config', 'src.preprocessing', 'src.features', 'src.predict']


## Build the preprocessed frame from raw CSVs

Calls `build_feature_frame` — applies all deterministic transforms (timestamps → durations, one-hots, ordinals, target binarization) and produces the single working frame `pp_df`. Columns are a superset of the locked feature list.

In [60]:
from src.preprocessing import build_feature_frame

pp_df = build_feature_frame(
    "data/raw/account_lifecycle_events.csv",
    "data/raw/user_engagement_metrics.csv",
    "data/raw/support_interaction_history.csv",
)
print("pp_df shape:", pp_df.shape)
pp_df['account_status'].value_counts(normalize=True).rename('class_balance')


pp_df shape: (5000, 142)


account_status
0    0.8908
1    0.1092
Name: class_balance, dtype: float64

## Apply realtime events

Demonstrates the realtime ingestion path: hand-craft a few events for a known
account_id and feed them through `apply_realtime_events`. New `rt_*` columns
and any in-place Class-A updates (`integration_count`, `seats_active`) appear
on the affected row. Downstream cells consume this realtime-augmented `pp_df`
— which is why cell 4 must rerun every time: the candidate column set has
shifted, and feature selection is the only gate that locks in which `rt_*`
columns are admitted on this run.

After `RETRAIN_EVERY_N_UPDATES` calls (configured in `src/config.py`), the API
schedules a full retrain as a fire-and-forget asyncio task: when the artifacts
on disk update, the served-prediction caches invalidate atomically.

In [ ]:
from src.api import apply_realtime_events, get_pp_df

ACCT = pp_df.index[0]  # pick any account that exists in the frame
before = pp_df.loc[ACCT, ["integration_count", "seats_active"]].to_dict()

events = [
    {"event_id": "e1", "event_type": "login",
     "account_id": ACCT, "user_id": "U_demo", "timestamp": "2026-05-18T12:00:00Z",
     "metadata": {"session_duration_minutes": 17}},
    {"event_id": "e2", "event_type": "integration_added",
     "account_id": ACCT, "timestamp": "2026-05-18T12:05:00Z", "metadata": {}},
    {"event_id": "e3", "event_type": "support_ticket_created",
     "account_id": ACCT, "timestamp": "2026-05-18T12:10:00Z",
     "metadata": {"category": "billing", "priority": "high", "sentiment": "negative"}},
]

status = await apply_realtime_events(events)
print("status:", status)

pp_df = get_pp_df()  # rebind to the realtime-augmented frame
rt_cols = [c for c in pp_df.columns if c.startswith("rt_")]
print(f"\n{len(rt_cols)} rt_* columns surfaced. Non-zero values on {ACCT}:")
touched = pp_df.loc[ACCT, rt_cols]
display(touched[touched != 0].rename("value").to_frame())
print("\nClass-A columns before vs after:")
print(" before:", before)
print(" after :", pp_df.loc[ACCT, ["integration_count", "seats_active"]].to_dict())

## Re-derive the locked feature list

This now runs every pass. Realtime events introduce or grow `rt_*` candidate columns,
and selection is the only gate that decides which enter the model — so the locked list
is no longer stable across runs.

`force=True` overwrites `models/feature_columns.json`.

In [61]:
from src.features import run_selection_pipeline

sel = run_selection_pipeline(pp_df, force=True)

print("locked features:", sel['locked_features'])
print()
print("stage drops:")
for stage, cols in sel['stage_drops'].items():
    print(f"  {stage}: {cols}")
print()
print("model-based AUC — l1: {l1:.4f}  rf: {rf:.4f}".format(**sel['auc']))
sel['rank_table'].head(20)

/Users/vidyutveedgav/churnguard-analytics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/vidyutveedgav/churnguard-analytics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/Users/vidyutveedgav/churnguard-analytics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified 

locked features: ['avg_dashboards_shared', 'avg_feature_usage_score', 'avg_logins_30d', 'days_since_contract_end', 'integration_count', 'rate_billing', 'rate_bug', 'rate_feature_request', 'rate_technical', 'seats_active', 'subscription_tier', 'total_logins_90d']

stage drops:
  leakage: ['health_score_band', 'risk_flag', 'days_since_last_activity', 'num_competitor_mentions', 'total_cancellation_requested', 'rate_cancellation_requested', 'num_cancellation', 'rate_cancellation', 'num_cancellation_reasons', 'total_retention_offer_made', 'total_account_pause_requested', 'total_downgrade_requested']
  variance: ['num_admins']
  correlation: ['avg_logins_90d', 'avg_sessions_30d', 'days_since_creation', 'days_since_earliest_resolution', 'days_since_latest_ticket_creation', 'days_since_latest_user_creation', 'max_feature_usage_score', 'max_reopened', 'min_feature_usage_score', 'mrr', 'num_active_users', 'num_distinct_agents', 'num_ticket_creators', 'num_tickets', 'num_users', 'rate_escalated',

,anova_F_rank,chi2_rank,mi_rank,mi,avg_rank
avg_feature_usage_score,1.0,NaN,1.0,0.071613,1.0
integration_count,2.0,NaN,2.0,0.068696,2.0
avg_reports_30d,3.0,NaN,3.0,0.043150,3.0
avg_dashboards_shared,5.0,NaN,5.0,0.036819,5.0
avg_session_duration_minutes,4.0,NaN,7.0,0.031272,5.5
avg_exports_30d,6.0,NaN,11.0,0.025280,8.5
avg_resolution_hours,10.0,NaN,8.0,0.027840,9.0
avg_logins_30d,8.0,NaN,10.0,0.025555,9.0
avg_dashboards_created,7.0,NaN,12.0,0.023971,9.5
total_dashboards_shared,13.0,NaN,6.0,0.033758,9.5


## Apply the locked feature list

Reads `models/feature_columns.json` and reindexes `pp_df` onto exactly those columns. Hard-fails (loud) if any locked column is missing or all-NaN — that's the schema-drift tripwire.

In [65]:
from src.features import apply_locked_features

X, y = apply_locked_features(pp_df)
print("X shape:", X.shape)
print("y balance:")
y.value_counts()

X shape: (5000, 12)
y balance:


account_status
0    4454
1     546
Name: count, dtype: int64

## Train, tune threshold, refit on full data, save artifacts

The four-call sequence that defines a "new model":

1. `train` — stratified split, fit on the train fold, return held-out probabilities
2. `tune_threshold` — F-beta sweep over thresholds, pick arg-max (β=2 weights recall 4× precision)
3. `fit_final` — refit on ALL labeled rows (spends the holdout for deployment)
4. `save_artifacts` — write `models/xgb_churn.json` + `models/config.json`


In [66]:
from src.predict import train, tune_threshold, fit_final, save_artifacts

fit   = train(X, y)
tuned = tune_threshold(
    fit['y_test'], fit['proba'],
    test_size=fit['test_size'],
    random_state=fit['random_state'],
)
final = fit_final(X, y)
save_artifacts(
    final,
    threshold=tuned['threshold'],
    holdout_metrics=tuned['holdout_metrics'],
    n_features=X.shape[1],
    n_training_rows=len(X),
)
print(f"saved — threshold={tuned['threshold']}  n_features={X.shape[1]}  n_rows={len(X)}")
tuned['sweep']


/Users/vidyutveedgav/churnguard-analytics/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:58:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vidyutveedgav/churnguard-analytics/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [13:58:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


saved — threshold=0.1  n_features=12  n_rows=5000


,threshold,flagged,TP,FP,FN,precision,recall,f1,f2
0,0.10,159,97,62,12,0.6101,0.8899,0.7239,0.8151
1,0.15,137,93,44,16,0.6788,0.8532,0.7561,0.8115
2,0.20,119,90,29,19,0.7563,0.8257,0.7895,0.8108
3,0.25,106,87,19,22,0.8208,0.7982,0.8093,0.8026
4,0.30,100,86,14,23,0.8600,0.7890,0.8230,0.8022
5,0.35,95,84,11,25,0.8842,0.7706,0.8235,0.7910
6,0.40,88,80,8,29,0.9091,0.7339,0.8122,0.7634
7,0.45,85,78,7,31,0.9176,0.7156,0.8041,0.7486
8,0.50,82,76,6,33,0.9268,0.6972,0.7958,0.7336
9,0.55,79,75,4,34,0.9494,0.6881,0.7979,0.7282


## Sanity check: reload artifacts from disk and predict

Round-trip test. If this cell produces sensible probabilities, the dashboard and `predict.py` consumers will work

In [67]:
import json
import pandas as pd
from xgboost import XGBClassifier

from src.config import MODEL_WEIGHTS_PATH, MODEL_CONFIG_PATH

reloaded = XGBClassifier()
reloaded.load_model(MODEL_WEIGHTS_PATH)
cfg = json.loads(MODEL_CONFIG_PATH.read_text())

sample = X.head(5)
proba = reloaded.predict_proba(sample)[:, 1]
flagged = (proba >= cfg['threshold']).astype(int)

print(f"loaded model: threshold={cfg['threshold']}  n_features={cfg['n_features']}  trained_at={cfg['trained_at']}")
pd.DataFrame({
    'proba':   proba.round(4),
    'flagged': flagged,
    'actual':  y.head(5).values,
}, index=sample.index)


loaded model: threshold=0.1  n_features=12  trained_at=2026-05-18T17:58:47+00:00


,proba,flagged,actual
account_id,,,
ACC000000,0.0008,0,0
ACC000001,0.0015,0,0
ACC000002,0.0017,0,0
ACC000003,0.0018,0,0
ACC000004,0.0058,0,0
